# 03 — Poster Embeddings (CLIP ViT-L/14)

Encodes poster images named `<tmdb_id>.jpg` into CLIP embeddings (open-clip ViT-L/14).
Saves per-id `.npy` and a `clip_index.csv`. Paths are read from env vars:
`IMG_DIR` (defaults to `~/boxoffice/data/img`), `ART_DIR` (defaults to
`~/boxoffice/artifacts`), `CLIP_DIR` (defaults to `$ART_DIR/clip_emb`), and `BATCH_SIZE`.


In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path

ART_DIR   = Path(os.getenv("ART_DIR",   "~/boxoffice/artifacts")).expanduser()
DATA_DIR  = Path(os.getenv("DATA_DIR",  "~/boxoffice/data")).expanduser()
IMG_DIR   = Path(os.getenv("IMG_DIR",   "~/boxoffice/data/img")).expanduser()

TEXT_DIR  = Path(os.getenv("TEXT_DIR",  ART_DIR / "text_emb")).expanduser()
CLIP_DIR  = Path(os.getenv("CLIP_DIR",  ART_DIR / "clip_emb")).expanduser()
ROI_DIR   = Path(os.getenv("ROI_DIR",   ART_DIR / "roi_feat")).expanduser()

def load_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

# Handy helper
def _vec(path, dim):
    try:
        v = np.load(path); v = np.asarray(v, np.float32).ravel()
        if   v.size == dim: return v
        elif v.size >  dim: return v[:dim]
        else:               return np.pad(v, (0, dim - v.size))
    except Exception:
        return np.zeros(dim, np.float32)


In [ ]:
## FOR RELOAD ONLY
clip_index = pd.read_csv(ART_DIR / "clip_index.csv")
clip_meta  = load_json(ART_DIR / "clip_model_meta.json")
# vec = np.load(CLIP_DIR / f"{tmdb_id}.npy")

In [8]:
import os as _os
_os.environ["TQDM_NOTEBOOK"] = "0"  # no ipywidgets
from tqdm import tqdm                
try:
    tqdm.monitor_interval = 0        
except Exception:
    pass

In [9]:
# ---- imports & params ----
import os, math, gc
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

IMG_DIR   = Path(os.getenv("IMG_DIR", os.path.expanduser("~/boxoffice/data/img")))
ART_DIR   = Path(os.getenv("ART_DIR", os.path.expanduser("~/boxoffice/artifacts")))
CLIP_DIR  = Path(os.getenv("CLIP_DIR", str(ART_DIR / "clip_emb")))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))

CLIP_DIR.mkdir(parents=True, exist_ok=True)

print("IMG_DIR:", IMG_DIR)
print("CLIP_DIR:", CLIP_DIR)
print("BATCH_SIZE:", BATCH_SIZE)

IMG_DIR: C:\Users\Vaishob\boxoffice\data\img
CLIP_DIR: C:\Users\Vaishob\boxoffice\artifacts\clip_emb
BATCH_SIZE: 64


In [10]:
# ---- CLIP (open-clip) ----
import torch
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "ViT-L-14"         # backbone per your choice
pretrained = "openai"           # weights tag in open-clip
print("Loading CLIP:", model_name, pretrained, "on", device)

model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
model.eval()

Loading CLIP: ViT-L-14 openai on cpu


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwi

In [11]:
# ---- discover poster ids from filenames <tmdb_id>.jpg ----
ids = []
for p in IMG_DIR.glob("*.jpg"):
    stem = p.stem
    try:
        ids.append(int(stem))
    except:
        continue

ids = sorted(set(ids))
print("Found posters:", len(ids))

Found posters: 4177


In [12]:
# ---- helpers ----
def l2_normalize(x, eps=1e-9):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return (x / np.maximum(n, eps)).astype(np.float32)

def load_image(path):
    img = Image.open(path).convert("RGB")
    return img

import torch
@torch.no_grad()
def encode_batch(img_list):
    if not img_list:
        return None
    ims = [preprocess(im).unsqueeze(0) for im in img_list]
    ims = torch.cat(ims, dim=0).to(device)
    feats = model.encode_image(ims)
    feats = feats.float().cpu().numpy()
    feats = l2_normalize(feats)
    return feats

In [13]:
# ---- encode & save ----
index_rows = []
batch_imgs, batch_ids = [], []

for tmdb_id in tqdm(ids):
    img_path = IMG_DIR / f"{tmdb_id}.jpg"
    if not img_path.exists():
        continue
    try:
        im = load_image(img_path)
        batch_imgs.append(im)
        batch_ids.append(tmdb_id)
        if len(batch_imgs) >= BATCH_SIZE:
            feats = encode_batch(batch_imgs)
            for tid, vec in zip(batch_ids, feats):
                np.save(CLIP_DIR / f"{tid}.npy", vec)
                index_rows.append({"tmdb_id": int(tid), "dim": int(vec.shape[0])})
            batch_imgs, batch_ids = [], []
            gc.collect()
    except Exception as e:
        print("Skip", tmdb_id, "error:", e)

# flush remainder
if batch_imgs:
    feats = encode_batch(batch_imgs)
    for tid, vec in zip(batch_ids, feats):
        np.save(CLIP_DIR / f"{tid}.npy", vec)
        index_rows.append({"tmdb_id": int(tid), "dim": int(vec.shape[0])})
    batch_imgs, batch_ids = [], []
    gc.collect()

pd.DataFrame(index_rows).to_csv(CLIP_DIR / "clip_index.csv", index=False)
print("Saved embeddings:", len(index_rows))

100%|████████████████████████████████████████████████████████████████████████████| 4177/4177 [2:08:42<00:00,  1.85s/it]


Saved embeddings: 4177


In [15]:
from glob import glob
clip_ids = sorted(int(Path(p).stem) for p in glob(str(CLIP_DIR / "*.npy")))
pd.DataFrame({"tmdb_id": clip_ids, "dim":[768]*len(clip_ids)}).to_csv(ART_DIR / "clip_index.csv", index=False)
save_json({"model":"open_clip ViT-L/14","pretrained":"openai"}, ART_DIR / "clip_model_meta.json")
